# Phase 8.3 — Decision Tree: Violent vs Non-violent Incident

**Goal:** classify a single recorded incident as violent (`is_violent = 1`) or non-violent (`0`) from when and where it happened.

**Input:** `data/processed/incidents.parquet` — one row per incident with a known time (time-unknown records are excluded).
**Script:** `ml/train_tree.py` (this notebook reads its saved results; it does not retrain).

## 1. Method

- **Features (Master Plan):** `hour`, `day_of_week`, `season`, `location_group`, `domestic`, `community_area`, `district`.
- **Deliberately excluded:** `primary_type`, `crime_group`, `iucr`, `fbi_code`, `description` (they define what "violent" means, so using them would leak the answer), plus `month`, `year`, `date`.
- **Time-based split:** train = 2023–2024, test = 2025 onwards. Records with an unknown time are removed first, because their `hour` is unreliable.
- **Model:** decision tree, `max_depth = 6`, `min_samples_leaf = 500` (keeps leaves stable and the tree readable).
- **Comparisons:** a majority-class baseline ("always non-violent") and a second tree with `class_weight = balanced`.
- **Metrics:** about a third of incidents are violent, so accuracy alone is misleading. We also report precision, recall and F1 for the violent class and the confusion matrix.

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

BASE = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
RESULTS, MODELS, SHOTS = f"{BASE}/ml/results", f"{BASE}/ml/models", f"{BASE}/docs/screenshots"
pd.set_option("display.width", 200)

## 2. Input data

In [ ]:
inc = pd.read_parquet(f"{BASE}/data/processed/incidents.parquet")
print("Shape:", inc.shape, "| missing cells:", int(inc.isna().sum().sum()))
print(inc.groupby("split")["is_violent"].agg(rows="size", violent_share="mean").round(3))
print("\nFeatures used by the model: hour, domestic, day_of_week, season, location_group, community_area, district")

## 3. Results on the test period (2025 onwards)

In [ ]:
m = pd.read_csv(f"{RESULTS}/tree_metrics.csv")
display(m.round(4))

In [ ]:
cm = pd.read_csv(f"{RESULTS}/tree_confusion_matrix.csv", index_col=0)
display(cm)
tn, fp = cm.iloc[0, 0], cm.iloc[0, 1]
fn, tp = cm.iloc[1, 0], cm.iloc[1, 1]
print(f"Violent incidents correctly found (true positives): {tp:,}")
print(f"Violent incidents missed (false negatives)        : {fn:,}  ({fn/(fn+tp)*100:.1f}% of all violent incidents)")
print(f"Non-violent flagged as violent (false positives)  : {fp:,}")
display(Image(f"{SHOTS}/13_tree_confusion_matrix.png"))

## 4. What the tree uses

In [ ]:
imp = pd.read_csv(f"{RESULTS}/tree_feature_importance.csv")
grouped = imp.groupby("original_feature")["importance"].sum().sort_values(ascending=False)
display((grouped * 100).round(2).rename("importance %").to_frame())
display(Image(f"{SHOTS}/15_tree_feature_importance.png"))
display(Image(f"{SHOTS}/14_tree_visualization.png"))

### Why does `domestic` dominate?
Domestic incidents are mostly battery and assault, which count as violent, so the tree mostly learns "domestic flag → violent". The check below shows how strongly the two are linked in both periods.

In [ ]:
chk = inc.groupby(["split", "domestic"])["is_violent"].agg(rows="size", violent_share="mean").round(3)
display(chk)

## 5. Using the saved model (what the API does)

In [ ]:
store = joblib.load(f"{MODELS}/tree.pkl")
print("Features:", store["features"])
print("max_depth:", store["max_depth"], "| min_samples_leaf:", store["min_samples_leaf"], "| class_weight:", store["class_weight"])
print("Train:", store["train_date_range"], "| Test:", store["test_date_range"])

test = inc[inc["split"] == "test"]
pred = store["pipeline"].predict(test[store["features"]])
print("\nRecomputed test accuracy:", round((pred == test["is_violent"]).mean(), 4))
print("Recomputed violent recall:", round(((pred == 1) & (test["is_violent"] == 1)).sum() / (test["is_violent"] == 1).sum(), 4))

example = pd.DataFrame([{"hour": 22, "domestic": 0, "day_of_week": 5, "season": "SUMMER",
                         "location_group": "STREET", "community_area": 25, "district": 15}])
p = store["pipeline"].predict_proba(example[store["features"]])[0, 1]
print(f"\nExample (Saturday 22:00, summer, street, not domestic): probability violent = {p:.2f}")

## 6. Interpretation and limitations

- **The tree beats the baseline, but only modestly.** Accuracy is 73.2% against 66.4% for "always non-violent", and balanced accuracy is 0.65 (0.50 is chance).
- **Violent incidents are often missed.** Recall for the violent class is 40.4% (precision 66.7%, F1 0.50): about 75,000 violent incidents in the test set are classified as non-violent. When it predicts "violent" it is right about two times in three.
- **The balanced tree trades precision for recall:** recall rises to 44.0% and F1 to 0.52, but precision falls to 62.4% and accuracy to 72.3%. The plain tree is the one saved for the API.
- **Feature importance:** `domestic` carries about 82% of the importance, then `location_group` (school, street, transport) and `hour`. Season, day of week and area contribute little. This describes how the fitted tree splits the data; it does not mean domestic status *causes* violence.
- **Why scores are limited:** without the crime type, only when/where/domestic information is available, and these features do not separate violent from non-violent incidents very well. That is expected and is the reason crime-type columns are excluded.
- **Use:** the output is a probability for a hypothetical incident, not a prediction of any specific crime or person. Results describe recorded incidents only.